# Check which COMP IDs exist in placenta only, cord only and common in both

# Imports & Load Data

In [2]:
import sys
from pathlib import Path
import os
import pandas as pd

sys.path.insert(0, str(Path(os.getcwd()).parent.parent))

from src.utils.config import (
    PLACENTA_ANNO, CORD_ANNO,
    OUTPUTS_DIR
)
from src.utils.io_utils import save_csv  # if needed



# Load Annotation Files

In [3]:

placenta_anno = pd.read_csv(PLACENTA_ANNO)
cord_anno = pd.read_csv(CORD_ANNO)

print("Annotation Data Loaded:")
print(f"  Placenta: {placenta_anno.shape[0]} metabolites")
print(f"  Cord: {cord_anno.shape[0]} metabolites")
print("\nFirst few COMP IDs:")
print("Placenta:", placenta_anno.iloc[:, 0][:3].tolist())
print("Cord:", cord_anno.iloc[:, 0][:3].tolist())

Annotation Data Loaded:
  Placenta: 897 metabolites
  Cord: 981 metabolites

First few COMP IDs:
Placenta: ['1002002', '1004006', '107004']
Cord: ['107004', '108005', '109011']


# Extract & Compare COMP IDs


In [4]:
# Extract COMP ID (column 0) as strings
placenta_comp_ids = set(placenta_anno.iloc[:, 0].astype(str))
cord_comp_ids = set(cord_anno.iloc[:, 0].astype(str))

print(f"Total Metabolites in Placenta (Unique COMP IDs in placenta annotation) : ")
print(f"  Placenta: {len(placenta_comp_ids)}")

print(f"Total Metabolites in Cord (Unique COMP IDs in cord annotation) : ")
print(f"  Cord: {len(cord_comp_ids)}")

# Find intersections/unions (Common, Chord Only, Placenta Only)
common = placenta_comp_ids.intersection(cord_comp_ids)
cord_only = cord_comp_ids - placenta_comp_ids
placenta_only = placenta_comp_ids - cord_comp_ids

print(f"\n Results:")
print(f"  Common: Metabolites found in both : {len(common)}")
print(f"  Metabolites found in Cord only: {len(cord_only)}")
print(f"  Metabolites found in Placenta only: {len(placenta_only)}")


Total Metabolites in Placenta (Unique COMP IDs in placenta annotation) : 
  Placenta: 897
Total Metabolites in Cord (Unique COMP IDs in cord annotation) : 
  Cord: 981

 Results:
  Common: Metabolites found in both : 698
  Metabolites found in Cord only: 283
  Metabolites found in Placenta only: 199


# Create Summary DataFrames

In [5]:
# Convert sets back to DataFrames with full annotation
def filter_anno(df, comp_id_set, comp_col=0):
    """Filter annotation by COMP ID set"""
    mask = df.iloc[:, comp_col].astype(str).isin(comp_id_set)
    return df[mask].copy()

common_df = filter_anno(placenta_anno, common)  # or cord_anno, same COMP IDs
cord_only_df = filter_anno(cord_anno, cord_only)
placenta_only_df = filter_anno(placenta_anno, placenta_only)

print("Summary tables created:")
print(f"  Common: {common_df.shape}")
print(f"  Cord only: {cord_only_df.shape}")
print(f"  Placenta only: {placenta_only_df.shape}")


Summary tables created:
  Common: (698, 11)
  Cord only: (283, 11)
  Placenta only: (199, 11)


In [6]:
common_df.head()

,COMP ID,COMPOUND Name,SUPER META PATHWAY,SUB META PATHWAY,ACQUISITION METHOD,HMDB ID,HMDB_annotation,PUBCHEM ID,CAS ID,KEGG ID,CHEMSPIDER ID
2,107004,beta-citrylglutamate,Amino Acid,Glutamate Metabolism,NEGa,HMDB0013220,NaN,NaN,NaN,NaN,NaN
4,108005,5-oxoproline,Amino Acid,Glutathione Metabolism,NEGa,HMDB0000267,NaN,7405.0,98-79-3,C01879,NaN
7,109011,N-acetylthreonine,Amino Acid,"Glycine, Serine and Threonine Metabolism",NEGa,HMDB0062557,NaN,152204.0,17093-74-2,NaN,NaN
8,1101006,tartronate (hydroxymalonate),Xenobiotics,Food Component/Plant,NEGa,HMDB0035227,NaN,45.0,80-69-3,C02287,44.0
9,1102005,2-hydroxyhippurate (salicylurate),Xenobiotics,Benzoate Metabolism,NEGa,HMDB0000840,NaN,10253.0,487-54-7,C07588,9835.0


In [7]:
cord_only_df.head()

,COMP ID,COMPOUND Name,SUPER META PATHWAY,SUB META PATHWAY,ACQUISITION METHOD,HMDB ID,HMDB_annotation,PUBCHEM ID,CAS ID,KEGG ID,CHEMSPIDER ID
3,1101001,1H-indole-7-acetic acid,Xenobiotics,Bacterial/Fungal,NEGa,NaN,NaN,NaN,NaN,NaN,NaN
6,1102011,3-(3-hydroxyphenyl)propionate,Xenobiotics,Benzoate Metabolism,NEGa,HMDB0000375,NaN,91.0,621-54-5,C11457,89.0
7,1102012,3-(3-hydroxyphenyl)propionate sulfate,Xenobiotics,Benzoate Metabolism,NEGa,HMDB0094710,NaN,187488.0,NaN,NaN,NaN
9,1102018,3-hydroxyhippurate sulfate,Xenobiotics,Benzoate Metabolism,NEGa,NaN,NaN,NaN,NaN,NaN,NaN
11,1102020,3-methoxycatechol sulfate (2),Xenobiotics,Benzoate Metabolism,NEGa,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
placenta_only_df.head()

,COMP ID,COMPOUND Name,SUPER META PATHWAY,SUB META PATHWAY,ACQUISITION METHOD,HMDB ID,HMDB_annotation,PUBCHEM ID,CAS ID,KEGG ID,CHEMSPIDER ID
0,1002002,gentisic acid-5-glucoside,Secondary metabolism,Benzenoids,NEGa,NaN,NaN,NaN,NaN,NaN,NaN
1,1004006,feruloylquinate (4),Secondary metabolism,Phenylpropanoids,NEGa,NaN,NaN,NaN,NaN,NaN,NaN
3,108004,4-hydroxy-nonenal-glutathione,Amino Acid,Glutathione Metabolism,NEGa,NaN,NaN,NaN,NaN,NaN,NaN
5,108015,"S-(1,2-dicarboxyethyl)glutathione",Amino Acid,Glutathione Metabolism,NEGa,NaN,NaN,NaN,NaN,NaN,NaN
6,109002,3-phosphoserine,Amino Acid,"Glycine, Serine and Threonine Metabolism",NEGa,HMDB0000272,"HMDB0000272,HMDB0001721(DL-O-Phosphoserine)",NaN,NaN,C01005,NaN


# Save Excel with 3 Sheets

In [13]:
summary_file = os.path.join(OUTPUTS_DIR, "summary_tables", "metabolites_common_cord_placenta.xlsx")

with pd.ExcelWriter(summary_file, engine='openpyxl') as writer:
    common_df.to_excel(writer, sheet_name='Common', index=False)
    cord_only_df.to_excel(writer, sheet_name='Cord_Only', index=False)
    placenta_only_df.to_excel(writer, sheet_name='Placenta_Only', index=False)

print(f"✅ Saved: {summary_file}")
print("\nSheets created:")
print("  1. Common")
print("  2. Cord_Only") 
print("  3. Placenta_Only")


✅ Saved: /Users/aishvarya/Library/CloudStorage/OneDrive-UniversityatBuffalo/Desktop/Project/Metabolomics/Metabolomics_Project/src/../outputs/summary_tables/metabolites_common_cord_placenta.xlsx

Sheets created:
  1. Common
  2. Cord_Only
  3. Placenta_Only


# Preview Results

In [23]:
common_df.head()

,COMP ID,COMPOUND Name,SUPER META PATHWAY,SUB META PATHWAY,ACQUISITION METHOD,HMDB ID,HMDB_annotation,PUBCHEM ID,CAS ID,KEGG ID,CHEMSPIDER ID
2,107004,beta-citrylglutamate,Amino Acid,Glutamate Metabolism,NEGa,HMDB0013220,NaN,NaN,NaN,NaN,NaN
4,108005,5-oxoproline,Amino Acid,Glutathione Metabolism,NEGa,HMDB0000267,NaN,7405.0,98-79-3,C01879,NaN
7,109011,N-acetylthreonine,Amino Acid,"Glycine, Serine and Threonine Metabolism",NEGa,HMDB0062557,NaN,152204.0,17093-74-2,NaN,NaN
8,1101006,tartronate (hydroxymalonate),Xenobiotics,Food Component/Plant,NEGa,HMDB0035227,NaN,45.0,80-69-3,C02287,44.0
9,1102005,2-hydroxyhippurate (salicylurate),Xenobiotics,Benzoate Metabolism,NEGa,HMDB0000840,NaN,10253.0,487-54-7,C07588,9835.0


In [18]:
cord_only_df.head()

,COMP ID,COMPOUND Name,SUPER META PATHWAY,SUB META PATHWAY,ACQUISITION METHOD,HMDB ID,HMDB_annotation,PUBCHEM ID,CAS ID,KEGG ID,CHEMSPIDER ID
3,1101001,1H-indole-7-acetic acid,Xenobiotics,Bacterial/Fungal,NEGa,NaN,NaN,NaN,NaN,NaN,NaN
6,1102011,3-(3-hydroxyphenyl)propionate,Xenobiotics,Benzoate Metabolism,NEGa,HMDB0000375,NaN,91.0,621-54-5,C11457,89.0
7,1102012,3-(3-hydroxyphenyl)propionate sulfate,Xenobiotics,Benzoate Metabolism,NEGa,HMDB0094710,NaN,187488.0,NaN,NaN,NaN
9,1102018,3-hydroxyhippurate sulfate,Xenobiotics,Benzoate Metabolism,NEGa,NaN,NaN,NaN,NaN,NaN,NaN
11,1102020,3-methoxycatechol sulfate (2),Xenobiotics,Benzoate Metabolism,NEGa,NaN,NaN,NaN,NaN,NaN,NaN


In [19]:
placenta_only_df.head()

,COMP ID,COMPOUND Name,SUPER META PATHWAY,SUB META PATHWAY,ACQUISITION METHOD,HMDB ID,HMDB_annotation,PUBCHEM ID,CAS ID,KEGG ID,CHEMSPIDER ID
0,1002002,gentisic acid-5-glucoside,Secondary metabolism,Benzenoids,NEGa,NaN,NaN,NaN,NaN,NaN,NaN
1,1004006,feruloylquinate (4),Secondary metabolism,Phenylpropanoids,NEGa,NaN,NaN,NaN,NaN,NaN,NaN
3,108004,4-hydroxy-nonenal-glutathione,Amino Acid,Glutathione Metabolism,NEGa,NaN,NaN,NaN,NaN,NaN,NaN
5,108015,"S-(1,2-dicarboxyethyl)glutathione",Amino Acid,Glutathione Metabolism,NEGa,NaN,NaN,NaN,NaN,NaN,NaN
6,109002,3-phosphoserine,Amino Acid,"Glycine, Serine and Threonine Metabolism",NEGa,HMDB0000272,"HMDB0000272,HMDB0001721(DL-O-Phosphoserine)",NaN,NaN,C01005,NaN
